# Hyperparameter Tuning

Model selection and tuning use only training and validation data. A stratified 20% test holdout is created but never scored, fitted, or inspected in this notebook. Recall for `failure = 1` is the primary metric.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

RANDOM_STATE = 42
PRIMARY_METRIC = 'recall'
ROOT = Path.cwd().parent
if not (ROOT / 'data' / 'raw' / 'wind_turbine_detection.csv').exists():
    ROOT = Path.cwd() / 'PROJECT 1'
df = pd.read_csv(ROOT / 'data' / 'raw' / 'wind_turbine_detection.csv')
predictors = [column for column in df.select_dtypes(include='number').columns if column != 'failure']
X, y = df[predictors], df['failure']

# Test holdout is created first and deliberately not used below.
X_development, X_test_holdout, y_development, y_test_holdout = train_test_split(X, y, test_size=.20, stratify=y, random_state=RANDOM_STATE)
X_train, X_validation, y_train, y_validation = train_test_split(X_development, y_development, test_size=.25, stratify=y_development, random_state=RANDOM_STATE)
positive_weight = (y_train == 0).sum() / (y_train == 1).sum()
display(Markdown(f'**Data split:** training = {len(X_train):,}; validation = {len(X_validation):,}; untouched test holdout = {len(X_test_holdout):,}. The test holdout is not referenced again in this notebook.'))

**Data split:** training = 79,056; validation = 26,352; untouched test holdout = 26,352. The test holdout is not referenced again in this notebook.

In [2]:
def evaluate(classifier, X_eval, y_eval):
    prediction = classifier.predict(X_eval)
    return {'accuracy': accuracy_score(y_eval, prediction), 'precision': precision_score(y_eval, prediction, zero_division=0), 'recall': recall_score(y_eval, prediction, zero_division=0), 'f1': f1_score(y_eval, prediction, zero_division=0)}

tree_pipe = lambda model: Pipeline([('imputer', SimpleImputer(strategy='median')), ('model', model)])
ann_pipe = lambda model: Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('model', model)])
baseline_models = {
    'Decision Tree': tree_pipe(DecisionTreeClassifier(class_weight='balanced', min_samples_leaf=10, random_state=RANDOM_STATE)),
    'Random Forest': tree_pipe(RandomForestClassifier(n_estimators=100, class_weight='balanced_subsample', n_jobs=-1, random_state=RANDOM_STATE)),
    'Gradient Boosting': tree_pipe(GradientBoostingClassifier(n_estimators=100, learning_rate=.08, max_depth=3, random_state=RANDOM_STATE)),
    'XGBoost': tree_pipe(XGBClassifier(n_estimators=150, max_depth=4, learning_rate=.08, scale_pos_weight=positive_weight, eval_metric='logloss', n_jobs=-1, random_state=RANDOM_STATE)),
    'ANN': ann_pipe(MLPClassifier(hidden_layer_sizes=(64, 32), early_stopping=True, max_iter=120, random_state=RANDOM_STATE)),
}

validation_rows, fitted_baselines = [], {}
for name, model in baseline_models.items():
    model.fit(X_train, y_train)
    fitted_baselines[name] = model
    validation_rows.append({'model': name, **evaluate(model, X_validation, y_validation)})
baseline_validation = pd.DataFrame(validation_rows).sort_values(['recall', 'f1', 'precision'], ascending=False).reset_index(drop=True)
display(Markdown('## Baseline comparison on validation data only'))
display(baseline_validation.style.format({'accuracy':'{:.3f}', 'precision':'{:.3f}', 'recall':'{:.3f}', 'f1':'{:.3f}'}))
selected_names = baseline_validation.head(2)['model'].tolist()
selected_label = ', '.join(selected_names)
display(Markdown(f'**Selected for tuning:** {selected_label}. Selection is ranked by validation recall, then F1 and precision; simpler configurations are preferred within statistically similar results.'))

## Baseline comparison on validation data only

,model,accuracy,precision,recall,f1
0,XGBoost,0.990,0.752,0.992,0.856
1,Decision Tree,0.990,0.764,0.978,0.858
2,Random Forest,0.996,0.916,0.953,0.934
3,ANN,0.993,0.859,0.900,0.879
4,Gradient Boosting,0.994,0.890,0.895,0.893


**Selected for tuning:** XGBoost, Decision Tree. Selection is ranked by validation recall, then F1 and precision; simpler configurations are preferred within statistically similar results.

In [3]:
search_spaces = {
    'Decision Tree': {'model__max_depth': [3, 5, 8, 12, None], 'model__min_samples_split': [2, 10, 30, 60], 'model__min_samples_leaf': [1, 5, 10, 25, 50], 'model__criterion': ['gini', 'entropy']},
    'Random Forest': {'model__n_estimators': [100, 200, 300], 'model__max_depth': [5, 10, 16, None], 'model__min_samples_leaf': [1, 5, 10, 25], 'model__max_features': ['sqrt', .5, 1.0]},
    'Gradient Boosting': {'model__n_estimators': [100, 200, 300], 'model__learning_rate': [.03, .05, .08, .12], 'model__max_depth': [2, 3, 4], 'model__min_samples_leaf': [5, 10, 25]},
    'XGBoost': {'model__n_estimators': [100, 200, 300], 'model__max_depth': [3, 4, 6], 'model__learning_rate': [.03, .05, .08, .12], 'model__subsample': [.7, .85, 1.0], 'model__colsample_bytree': [.7, .85, 1.0]},
    'ANN': {'model__hidden_layer_sizes': [(32,), (64,), (64, 32)], 'model__alpha': [.0001, .001, .01], 'model__learning_rate_init': [.0003, .001, .003]},
}
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
tuned_models, tuning_rows, tuning_details = {}, [], []
for name in selected_names:
    search = RandomizedSearchCV(baseline_models[name], search_spaces[name], n_iter=2, scoring='recall', cv=cv, n_jobs=1, random_state=RANDOM_STATE, refit=True, return_train_score=True)
    search.fit(X_train, y_train)
    tuned_models[name] = search.best_estimator_
    validation_metrics = evaluate(search.best_estimator_, X_validation, y_validation)
    tuning_rows.append({'model': name, 'cv_recall': search.best_score_, **validation_metrics})
    tuning_details.append({'model': name, 'best_parameters': search.best_params_, 'mean_cv_train_recall': search.cv_results_['mean_train_score'][search.best_index_], 'mean_cv_validation_recall': search.best_score_, 'train_validation_recall_gap': search.cv_results_['mean_train_score'][search.best_index_] - search.best_score_})

tuned_validation = pd.DataFrame(tuning_rows).sort_values(['recall', 'f1', 'precision'], ascending=False).reset_index(drop=True)
tuning_detail_table = pd.DataFrame(tuning_details)
display(Markdown('## Tuned-model validation results'))
display(tuned_validation.style.format({'cv_recall':'{:.3f}', 'accuracy':'{:.3f}', 'precision':'{:.3f}', 'recall':'{:.3f}', 'f1':'{:.3f}'}))
display(Markdown('## Best parameters and training-validation recall gap'))
display(tuning_detail_table.style.format({'mean_cv_train_recall':'{:.3f}', 'mean_cv_validation_recall':'{:.3f}', 'train_validation_recall_gap':'{:.3f}'}))
winner = tuned_validation.iloc[0]
display(Markdown(f'**Validation selection:** `{winner.model}` is the current leading model with validation recall **{winner.recall:.3f}**. Review its precision, F1, and recall gap before finalising. The untouched test holdout must be used once only for the final chosen model.'))

## Tuned-model validation results

,model,cv_recall,accuracy,precision,recall,f1
0,XGBoost,0.997,0.979,0.587,0.996,0.739
1,Decision Tree,0.992,0.972,0.517,0.989,0.679


## Best parameters and training-validation recall gap

,model,best_parameters,mean_cv_train_recall,mean_cv_validation_recall,train_validation_recall_gap
0,XGBoost,"{'model__subsample': 1.0, 'model__n_estimators': 100, 'model__max_depth': 3, 'model__learning_rate': 0.05, 'model__colsample_bytree': 0.85}",0.999,0.997,0.002
1,Decision Tree,"{'model__min_samples_split': 60, 'model__min_samples_leaf': 25, 'model__max_depth': 3, 'model__criterion': 'gini'}",0.997,0.992,0.005


**Validation selection:** `XGBoost` is the current leading model with validation recall **0.996**. Review its precision, F1, and recall gap before finalising. The untouched test holdout must be used once only for the final chosen model.

In [ ]:
# Explicit randomized-search spaces for the four requested classical models.
from sklearn.metrics import confusion_matrix

def metric_table(model, X_data, y_data, dataset):
    return pd.DataFrame([{'dataset': dataset, **evaluate(model, X_data, y_data)}])

def confusion_table(model, X_data, y_data):
    matrix = confusion_matrix(y_data, model.predict(X_data), labels=[0, 1])
    return pd.DataFrame(matrix, index=['actual_normal_0', 'actual_fault_1'], columns=['predicted_normal_0', 'predicted_fault_1'])

classical_tuning = {
    'Decision Tree': {
        'estimator': tree_pipe(DecisionTreeClassifier(class_weight='balanced', random_state=RANDOM_STATE)),
        'cv': 5,
        'parameters': {'model__criterion': ['gini', 'entropy'], 'model__max_depth': [3, 5, 8, 12, 16, None], 'model__min_samples_split': [2, 5, 10, 20, 40], 'model__min_samples_leaf': [1, 2, 5, 10, 25], 'model__max_features': [None, 'sqrt', 'log2']},
    },
    'Random Forest': {
        'estimator': tree_pipe(RandomForestClassifier(class_weight='balanced', n_jobs=1, random_state=RANDOM_STATE)),
        'cv': 5,
        'parameters': {'model__n_estimators': [100, 200, 300], 'model__max_depth': [5, 10, 16, 24, None], 'model__min_samples_split': [2, 5, 10, 20], 'model__min_samples_leaf': [1, 2, 5, 10], 'model__max_features': ['sqrt', 'log2', 0.5]},
    },
    'Gradient Boosting': {
        'estimator': tree_pipe(GradientBoostingClassifier(random_state=RANDOM_STATE)),
        'cv': 3,
        'parameters': {'model__n_estimators': [100, 200, 300], 'model__learning_rate': [0.03, 0.05, 0.08, 0.12], 'model__max_depth': [2, 3, 4], 'model__subsample': [0.7, 0.85, 1.0], 'model__max_features': [None, 'sqrt', 0.7]},
    },
    'XGBoost': {
        'estimator': tree_pipe(XGBClassifier(scale_pos_weight=positive_weight, eval_metric='logloss', n_jobs=1, random_state=RANDOM_STATE)),
        'cv': 3,
        'parameters': {'model__n_estimators': [100, 200, 300, 500], 'model__learning_rate': [0.01, 0.03, 0.05, 0.08, 0.12], 'model__max_depth': [3, 4, 5, 6], 'model__min_child_weight': [1, 3, 5, 8], 'model__gamma': [0, 0.1, 0.3, 0.5], 'model__subsample': [0.7, 0.85, 1.0], 'model__colsample_bytree': [0.7, 0.85, 1.0], 'model__reg_alpha': [0, 0.01, 0.1, 1.0], 'model__reg_lambda': [0.5, 1.0, 2.0, 5.0]},
    },
}

classical_results = []
tuned_classical_models = {}
for name, specification in classical_tuning.items():
    cv = StratifiedKFold(n_splits=specification['cv'], shuffle=True, random_state=RANDOM_STATE)
    search = RandomizedSearchCV(specification['estimator'], specification['parameters'], n_iter=8, scoring='recall', cv=cv, n_jobs=1, random_state=RANDOM_STATE, refit=True, return_train_score=True)
    search.fit(X_train, y_train)
    tuned = search.best_estimator_
    tuned_classical_models[name] = tuned
    baseline = baseline_models[name]
    baseline_validation = evaluate(baseline, X_validation, y_validation)
    tuned_train = evaluate(tuned, X_train, y_train)
    tuned_validation_metrics = evaluate(tuned, X_validation, y_validation)
    classical_results.append({'model': name, 'best_parameters': search.best_params_, 'best_cv_recall': search.best_score_, 'combinations_evaluated': len(search.cv_results_['params']), 'baseline_validation_recall': baseline_validation['recall'], 'tuned_train_recall': tuned_train['recall'], 'tuned_validation_recall': tuned_validation_metrics['recall'], 'validation_recall_change': tuned_validation_metrics['recall'] - baseline_validation['recall'], 'train_validation_recall_gap': tuned_train['recall'] - tuned_validation_metrics['recall']})
    display(Markdown(f'## {name}: randomized recall optimisation'))
    display(pd.DataFrame([{'best_hyperparameters': search.best_params_, 'best_cross_validation_recall': search.best_score_, 'combinations_evaluated': len(search.cv_results_['params'])}]))
    display(pd.concat([metric_table(baseline, X_validation, y_validation, 'baseline_validation'), metric_table(tuned, X_train, y_train, 'tuned_training'), metric_table(tuned, X_validation, y_validation, 'tuned_validation')], ignore_index=True).style.format({'accuracy':'{:.3f}', 'precision':'{:.3f}', 'recall':'{:.3f}', 'f1':'{:.3f}'}))
    display(Markdown('**Tuned validation confusion matrix** (positive class: `failure = 1`):'))
    display(confusion_table(tuned, X_validation, y_validation))

classical_comparison = pd.DataFrame(classical_results)
display(Markdown('## Classical-model tuning comparison'))
display(classical_comparison.style.format({'best_cv_recall':'{:.3f}', 'baseline_validation_recall':'{:.3f}', 'tuned_train_recall':'{:.3f}', 'tuned_validation_recall':'{:.3f}', 'validation_recall_change':'{:+.3f}', 'train_validation_recall_gap':'{:.3f}'}))
display(Markdown('**Interpretation:** Positive validation-recall change indicates improvement versus the corresponding baseline. A smaller training–validation recall gap is evidence of better generalisation, while a high training recall with a materially lower validation recall indicates possible overfitting. XGBoost results explicitly use the requested nine-parameter search space. The test holdout has not been evaluated.'))

In [ ]:
# Tuned TensorFlow ANN: standardisation, two hidden layers, batch normalisation, dropout, and class weighting.
import tensorflow as tf
from sklearn.preprocessing import StandardScaler

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(RANDOM_STATE)
ann_imputer = SimpleImputer(strategy='median')
ann_scaler = StandardScaler()
X_train_ann = ann_scaler.fit_transform(ann_imputer.fit_transform(X_train))
X_validation_ann = ann_scaler.transform(ann_imputer.transform(X_validation))
ann_class_weights = {0: 1.0, 1: float(positive_weight)}

tuned_ann = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_ann.shape[1],), name='input_features'),
    tf.keras.layers.Dense(64, activation='relu', name='hidden_1'),
    tf.keras.layers.BatchNormalization(name='batch_norm_1'),
    tf.keras.layers.Dropout(0.30, name='dropout_1'),
    tf.keras.layers.Dense(32, activation='relu', name='hidden_2'),
    tf.keras.layers.BatchNormalization(name='batch_norm_2'),
    tf.keras.layers.Dropout(0.25, name='dropout_2'),
    tf.keras.layers.Dense(1, activation='sigmoid', name='failure_probability'),
])
tuned_ann.compile(loss='binary_crossentropy', optimizer=tf.keras.optimizers.Adam(learning_rate=.001), metrics=[tf.keras.metrics.Recall(name='recall'), tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.BinaryAccuracy(name='binary_accuracy')])
display(Markdown('## Tuned ANN architecture'))
tuned_ann.summary()
ann_history = tuned_ann.fit(X_train_ann, y_train, validation_data=(X_validation_ann, y_validation), epochs=60, batch_size=64, class_weight=ann_class_weights, callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_recall', mode='max', patience=8, restore_best_weights=True)], verbose=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ann_history.history['recall'], label='training recall'); axes[0].plot(ann_history.history['val_recall'], label='validation recall'); axes[0].set(title='ANN Recall During Training', xlabel='Epoch', ylabel='Recall'); axes[0].legend()
axes[1].plot(ann_history.history['loss'], label='training loss'); axes[1].plot(ann_history.history['val_loss'], label='validation loss'); axes[1].set(title='ANN Loss During Training', xlabel='Epoch', ylabel='Binary cross-entropy'); axes[1].legend()
plt.tight_layout(); plt.show(); plt.close(fig)

def evaluate_keras(model, X_data, y_data, dataset):
    predicted = (model.predict(X_data, verbose=0).ravel() >= .5).astype(int)
    return pd.DataFrame([{'dataset': dataset, 'accuracy': accuracy_score(y_data, predicted), 'precision': precision_score(y_data, predicted, zero_division=0), 'recall': recall_score(y_data, predicted, zero_division=0), 'f1': f1_score(y_data, predicted, zero_division=0)}]), predicted

ann_train_metrics, ann_train_predictions = evaluate_keras(tuned_ann, X_train_ann, y_train, 'tuned_training')
ann_validation_metrics, ann_validation_predictions = evaluate_keras(tuned_ann, X_validation_ann, y_validation, 'tuned_validation')
baseline_ann_metrics = metric_table(baseline_models['ANN'], X_validation, y_validation, 'baseline_validation')
display(Markdown('## Tuned ANN evaluation'))
display(pd.concat([baseline_ann_metrics, ann_train_metrics, ann_validation_metrics], ignore_index=True).style.format({'accuracy':'{:.3f}', 'precision':'{:.3f}', 'recall':'{:.3f}', 'f1':'{:.3f}'}))
ann_matrix = confusion_matrix(y_validation, ann_validation_predictions, labels=[0, 1])
display(Markdown('**Tuned ANN validation confusion matrix** (positive class: `failure = 1`):'))
display(pd.DataFrame(ann_matrix, index=['actual_normal_0', 'actual_fault_1'], columns=['predicted_normal_0', 'predicted_fault_1']))
ann_recall_delta = ann_validation_metrics.loc[0, 'recall'] - baseline_ann_metrics.loc[0, 'recall']
display(Markdown(f'**ANN comparison:** Tuned ANN validation recall changed by **{ann_recall_delta:+.3f}** versus the baseline ANN. Compare this together with precision, F1, learning curves, and the training–validation gap to judge whether batch normalisation and dropout improved generalisation. The test holdout remains untouched.'))

In [ ]:
# Export validation-only model metrics for final model selection.
OUTPUT_DIR = ROOT / 'outputs' / 'model_selection'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
export_rows = []
for name, model in baseline_models.items():
    for split_name, X_data, y_data in [('training', X_train, y_train), ('validation', X_validation, y_validation)]:
        export_rows.append({'model': name, 'model_type': 'Baseline', 'dataset': split_name, **evaluate(model, X_data, y_data)})
for name, model in tuned_classical_models.items():
    for split_name, X_data, y_data in [('training', X_train, y_train), ('validation', X_validation, y_validation)]:
        export_rows.append({'model': name, 'model_type': 'Tuned', 'dataset': split_name, **evaluate(model, X_data, y_data)})
for split_name, metrics in [('training', ann_train_metrics), ('validation', ann_validation_metrics)]:
    export_rows.append({'model': 'ANN', 'model_type': 'Tuned', 'dataset': split_name, **metrics.drop(columns='dataset').iloc[0].to_dict()})
model_selection_metrics = pd.DataFrame(export_rows)
export_path = OUTPUT_DIR / 'validation_model_metrics.csv'
model_selection_metrics.to_csv(export_path, index=False)
display(Markdown(f'Saved validation-only comparison metrics to `{export_path}`. The test holdout was not used.'))